In [133]:

from pathlib import Path
import numpy as np
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from chemprop.featurizers.atom import MultiHotAtomFeaturizer
from rdkit.Chem.rdchem import HybridizationType
import pandas as pd
from rdkit import Chem
from chemprop import data, featurizers, models, nn
from ml_enhance import QuantumFPFileLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from typing import Iterable
import pandas as pd
from chemprop.data import MoleculeDatapoint, MoleculeDataset
from chemprop.featurizers import SimpleMoleculeMolGraphFeaturizer

In [118]:
# check whether the order of the smiles in my used DF matches the order of the files list
check = []
for df_id, file in zip(used_ids, used_files, strict=True):
    matches = re.findall(r"\d+", file.name)
    id = int(matches[1])

    check.append(id == df_id)

np.sum(check) # is correct

np.int64(8763)

In [146]:
df.query("id == 5")["solubility"].iloc[0]

np.float64(-2.6645485391)

In [2]:
from data_preprocess import *

In [3]:
df = pd.read_csv("../data/processed_dataset_wo_metals_w_even_more_qm2.csv")
used_ids: list[int] = df["id"].apply(round).to_list()

qfp_loader = QuantumFPFileLoader(Path("../data/QuantumFP/QFP_output"))
filelist: list[Path] = qfp_loader.list_output_files()
used_files = np.array(filter_files(filelist, used_ids))

outer_splits = pd.read_pickle("../hpc_splits.pkl")
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

config = {
    "use_atom_features": False,
    "use_bond_features": False,
    "use_mol_features": False,
    "use_custom_atom_featurizer": False,
    "use_custom_bond_featurizer": False,
    "n_rbf_centers": 10,
    "target_col": "solubility",
}

In [4]:
tr_idxs, tst_idxs = outer_splits[3]
outer_train_files = used_files[tr_idxs][:2]
outer_test_files = used_files[tst_idxs][:2]

In [5]:
df.filter(regex="atomic").columns

Index(['avg_atomic_dipole_norm', 'min_atomic_dipole_norm',
       'max_atomic_dipole_norm', 'std_atomic_dipole_norm',
       'avg_atomic_quadrupole_principal_invariant_2',
       'min_atomic_quadrupole_principal_invariant_2',
       'max_atomic_quadrupole_principal_invariant_2',
       'std_atomic_quadrupole_principal_invariant_2',
       'avg_atomic_quadrupole_principal_invariant_3',
       'min_atomic_quadrupole_principal_invariant_3',
       'max_atomic_quadrupole_principal_invariant_3',
       'std_atomic_quadrupole_principal_invariant_3',
       'avg_atomic_polarizability_mean', 'min_atomic_polarizability_mean',
       'max_atomic_polarizability_mean', 'std_atomic_polarizability_mean',
       'avg_atomic_polarizability_anisotropy',
       'min_atomic_polarizability_anisotropy',
       'max_atomic_polarizability_anisotropy',
       'std_atomic_polarizability_anisotropy', 'avg_atomic_sasa',
       'min_atomic_sasa', 'max_atomic_sasa', 'std_atomic_sasa',
       'min_atomic_fukui_minu

In [6]:
# outer_train_dataset, outer_test_dataset, outer_target_scaler = build_fold(
#     outer_train_files, outer_test_files, df, qfp_loader, **config
# )

In [7]:
for sdf in qfp_loader.stream_conformer_dataframe(used_files[3]):
    tdf = sdf

tdf.loc[0, ["original_smiles", "output_smiles"]].values
tdf.loc[6, ["original_smiles", "output_smiles"]].values

# np.array(tdf["partial_charge"].values.tolist())[:, :, 1]
# tdf[["original_smiles", "partial_charge"]].head(5)
# tdf["partial_charge"][0]
# tdf["original_smiles"][0]

<StringArray>
['[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]', '[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]']
Length: 2, dtype: string

In [8]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [9]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O-:12])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O-:12])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [10]:
'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

'[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H:19])[c:6]([C:7]([H:20])([H:21])[H:22])[c:8]1[O:9][C:10]([C:11]([O:12][H:25])=[O:13])([H:23])[H:24])([H:14])([H:15])[H:16]'

In [11]:
train_features = process_files(
    outer_train_files,
    qfp_loader,
    use_atom_features=True,
    use_bond_features=False,
    use_mol_features=False,
)
val_features = process_files(
    outer_test_files,
    qfp_loader,
    use_atom_features=True,
    use_bond_features=False,
    use_mol_features=False,
)

In [12]:
train_atom_df = train_features["atoms"]
val_atom_df = val_features["atoms"]

In [13]:
atomic_features: list[str] = [
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "atomic_sasa",
    "partial_charge",
]

In [14]:
n_rbf_centers = 10

train_atom_df, val_atom_df, _ = scale_features(train_atom_df, val_atom_df, atomic_features)
train_atom_df.head()

,original_smiles,atom,atomic_fukui_minus,atomic_fukui_plus,atomic_dipole_norm,atomic_quadrupole_principal_invariant_2,atomic_quadrupole_principal_invariant_3,atomic_polarizability_mean,atomic_polarizability_anisotropy,atomic_sasa,partial_charge
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,1,-2.014815,-1.538670,1.452548,0.717111,0.623741,0.030648,-0.264434,1.846262,3.243694
1,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,2,0.507144,0.873173,-0.279674,-0.741992,-0.330513,0.311980,-0.014145,-0.142409,-2.222976
2,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,3,0.727361,-0.146713,-1.085092,1.028012,0.724608,0.244517,-0.191260,-0.424245,1.295188
3,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,4,0.553774,0.275433,1.251219,0.137615,0.507487,0.221924,-0.149116,-0.550528,-0.533555
4,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,5,0.654434,0.354008,-1.301430,-0.707742,-0.660359,0.206764,-0.000476,0.576310,0.405453


In [15]:
train_atom_df, atom_rbf_cols = apply_rbf(train_atom_df, atomic_features, n_rbf_centers)
# val_atom_df, _ = apply_rbf(val_atom_df, atomic_features, n_rbf_centers)

In [16]:
train_atom_df.head()

,original_smiles,atom,atomic_fukui_minus_0,atomic_fukui_minus_1,atomic_fukui_minus_2,atomic_fukui_minus_3,atomic_fukui_minus_4,atomic_fukui_minus_5,atomic_fukui_minus_6,atomic_fukui_minus_7,...,partial_charge_0,partial_charge_1,partial_charge_2,partial_charge_3,partial_charge_4,partial_charge_5,partial_charge_6,partial_charge_7,partial_charge_8,partial_charge_9
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,1,2.331938e-01,8.588326e-01,0.833759,0.213360,0.014392,0.000256,0.000001,1.481877e-09,...,4.021472e-26,5.470900e-21,1.961883e-16,1.854506e-12,4.620872e-09,0.000003,5.254575e-04,2.398034e-02,2.884788e-01,9.147725e-01
1,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,2,9.710246e-09,5.545833e-06,0.000835,0.033133,0.346594,0.955696,0.694640,1.330885e-01,...,4.042791e-01,9.818977e-01,6.286253e-01,1.060860e-01,4.719165e-03,0.000055,1.710414e-07,1.393575e-10,2.992955e-14,1.694379e-18
2,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,3,8.899764e-10,7.895746e-07,0.000185,0.011383,0.184961,0.792244,0.894493,2.662170e-01,...,9.589322e-13,2.648566e-09,1.928300e-06,3.700657e-04,1.872078e-02,0.249637,8.774778e-01,8.130238e-01,1.985688e-01,1.278379e-02
3,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,4,5.925737e-09,3.715192e-06,0.000614,0.026747,0.307144,0.929702,0.741799,1.560161e-01,...,1.089057e-04,7.759756e-03,1.457425e-01,7.215477e-01,9.416394e-01,0.323925,2.937283e-02,7.020812e-04,4.423537e-06,7.346706e-09
4,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,5,1.995589e-09,1.530179e-06,0.000309,0.016478,0.231419,0.856708,0.836002,2.150416e-01,...,2.787126e-08,1.298872e-05,1.595573e-03,5.166633e-02,4.410007e-01,0.992228,5.884705e-01,9.199802e-02,3.791167e-03,4.118206e-05


In [17]:
def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for sdf in loader.stream_conformer_dataframe(file):
        sdf["gibbs_free_energy_300K"] = sdf["gibbs_free_energy"].map(lambda x: x[1][1])
        yield sdf

In [24]:
def get_info(file):
    for sdf in stream_conformer_df(file, qfp_loader):
        tdf = sdf[["original_smiles", "gibbs_free_energy_300K", *atomic_features]]
        object_cols = tdf.select_dtypes(include="object").columns.to_list()
        exploded_df = tdf.explode(object_cols)

        atoms_per_conformer = exploded_df.groupby("gibbs_free_energy_300K").size()

        n_unique_sizes = atoms_per_conformer.nunique()
        
        energies = exploded_df["gibbs_free_energy_300K"].unique()

        return {
            "original_smiles": tdf["original_smiles"].iloc[0],
            "id": sdf["id"].iloc[0],
            "n_conformers": len(atoms_per_conformer),
            "atom_counts": atoms_per_conformer.unique().tolist(),
            "energies": energies,
            "inconsistent": n_unique_sizes > 1,
        }

In [25]:
from ml_enhance import parallelize

results = parallelize(get_info, used_files, n_jobs=6)

In [96]:
results = []
for file in used_files:
    for sdf in stream_conformer_df(file, qfp_loader):
        tdf = sdf[["original_smiles", "gibbs_free_energy_300K", *atomic_features]]
        object_cols = tdf.select_dtypes(include="object").columns.to_list()
        exploded_df = tdf.explode(object_cols)

        atoms_per_conformer = exploded_df.groupby("gibbs_free_energy_300K").size()

        n_unique_sizes = atoms_per_conformer.nunique()

        results.append({
            "original_smiles": tdf["original_smiles"].iloc[0],
            "n_conformers": len(atoms_per_conformer),
            "atom_counts": atoms_per_conformer.unique().tolist(),
            "inconsistent": n_unique_sizes > 1,
        })
        
pd.DataFrame(results)

,original_smiles,n_conformers,atom_counts,inconsistent
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,1,[20],False
1,[Cl:1][c:2]1[c:3]([H:10])[c:4]([H:11])[c:5]([C...,1,[14],False
2,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,19,[33],False
3,[C:1]([c:2]1[c:3]([H:17])[c:4]([H:18])[c:5]([H...,7,"[24, 25]",True
4,[C:1]([O:2][C:3](=[O:4])[c:5]1[c:6]([H:28])[c:...,10,[34],False
...,...,...,...,...
8758,[C:1]([C:2]([c:3]1[c:4]([H:29])[c:5]([C:6]([c:...,22,[53],False
8759,[C:1]([C:2]([C:3]([H:35])([H:36])[H:37])([c:4]...,32,[59],False
8760,[C:1]([C:2]([c:3]1[c:4]([H:34])[c:5]([H:35])[c...,18,[56],False
8761,[C:1]([C:2]([C:3]([C:4]([C:5]([C:6]([C:7]([H:3...,32,[39],False


In [97]:
result_df = pd.DataFrame(results)

In [ ]:
result_df["lens"] = result_df["atom_counts"].apply(len)

result_df.query("inconsistent == True").sort_values('lens', ascending=False).head(20)

NameError: name 'result_df' is not defined

In [105]:
result_df.iloc[1898].original_smiles

'[O:1]([c:2]1[c:3]([H:21])[c:4]([O:5][H:22])[c:6]2[c:17]([c:18]1[O:19][H:27])[C:15](=[O:16])[c:14]1[c:9]([c:10]([H:23])[c:11]([H:24])[c:12]([H:25])[c:13]1[H:26])[C:7]2=[O:8])[H:20]'

In [ ]:
'[O:1]([c:2]1[c:3]([H:21])[c:4]([O:5][H:22])[c:6]2[c:17]([c:18]1[O:19][H:27])[C:15](=[O:16])[c:14]1[c:9]([c:10]([H:23])[c:11]([H:24])[c:12]([H:25])[c:13]1[H:26])[C:7]2=[O:8])[H:20]'

In [ ]:
for sdf in stream_conformer_df(used_files[0], qfp_loader):
    print(sdf)

Index(['id', 'original_smiles', 'output_smiles', 'energy',
       'atomization_energy', 'homo_lumo_gap', 'ionization_energy',
       'electron_affinity', 'chemical_potential', 'molecular_dipole',
       'molecular_dipole_norm', 'molecular_quadrupole',
       'molecular_quadrupole_principal_invariant_2',
       'molecular_quadrupole_principal_invariant_3',
       'molecular_polarizability', 'molecular_polarizability_mean',
       'molecular_polarizability_anisotropy', 'normal_modes',
       'normal_mode_frequencies', 'infrared_intensity', 'enthalpy',
       'gibbs_free_energy', 'entropy', 'zero_point_energy',
       'radius_of_gyration', 'molecular_volume', 'sterimol_L', 'sterimol_Bmin',
       'sterimol_Bmax', 'molecular_sasa', 'heat_capacity',
       'solvation_energy_water', 'solvation_energy_thf',
       'solvation_energy_cyclohexane', 'solvation_energy_dmso',
       'effective_coordination_number', 'partial_charge', 'atomic_fukui_minus',
       'atomic_fukui_plus', 'atomic_dipole',